In [1]:
import pandas as pd
import numpy as np


terrorism = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\Global Terrorism Dataset\globalterrorismdb_0718dist.csv", encoding="Windows-1252")

C:\Users\HP\AppData\Local\Temp\ipykernel_16664\551476585.py:5: DtypeWarning: Columns (4,6,31,33,61,62,63,76,79,90,92,94,96,114,115,121) have mixed types. Specify dtype option on import or set low_memory=False.
  terrorism = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\Global Terrorism Dataset\globalterrorismdb_0718dist.csv", encoding="Windows-1252")


In [2]:
replace_dict = {
    "unknown":pd.NA,
    "maoists":"communist party of india - maoist (cpi-maoist)",
    "palestinian extremists":"palestine liberation organization (plo)",
    "palestinians": "palestine liberation organization (plo)", 
    "muslim militants":"muslim extremists",
    "muslim separatists":"muslim extremists",
    "muslim fundamentalists":"muslim extremists",
    "muslim rebels":"muslim extremists",
    "muslims":"muslim extremists",
    "islamic state of iraq (isi)":"islamic state of iraq and the levant (isil)" 
}

In [3]:
#Run this cell only once. The conversion of unknown values to NaN leads to them being dropped on a second run.

terrorism["gname"] =(
    terrorism["gname"]
    .str.strip()
    .str.lower()
    .replace(to_replace=replace_dict)
) 


ranks = (
    terrorism
    .groupby("gname")["eventid"]
    .count()
    .sort_values(ascending=False)
)

ranks = ranks[ranks>30]
terrorism = terrorism[terrorism["gname"].isin(ranks.index)]

with open(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\ranks.txt", "w") as file_rank:
    file_rank.write(ranks.to_string())



In [4]:
default_col = [
    "target",
    "ransom",
    "targ",
    "corp",
    "natlty",
    "guncertain",
    "claimmode",
    "weap",
    "ishostkid",
    "scite",
    "hostkidoutcome",
    "prop",
    "nperp"
]
additional_columns = [
    "attacktype2_txt",
    "attacktype2",
    "attacktype3",
    "attacktype3_txt",
    "gsubname2",
    "gsubname3",
    "gname2",
    "gname3",
    "dbsource",
    "addnotes",
    "nkillus",
    "nwoundus",
    "nkillter",
    "nwoundte",
    "location" 
]

In [5]:

collist = [col for col in terrorism.columns if any(w in col.lower() for w in default_col)] + additional_columns
    

terrorism =terrorism.drop(columns=collist) 

terrorism = terrorism.drop(columns= [col for col in terrorism.columns if (terrorism[col].isna().mean())>0.85])
terrorism.columns

Index(['eventid', 'iyear', 'imonth', 'iday', 'extended', 'country',
       'country_txt', 'region', 'region_txt', 'provstate', 'city', 'latitude',
       'longitude', 'specificity', 'vicinity', 'summary', 'crit1', 'crit2',
       'crit3', 'doubtterr', 'alternative', 'alternative_txt', 'multiple',
       'success', 'suicide', 'attacktype1', 'attacktype1_txt', 'gname',
       'motive', 'individual', 'claimed', 'nkill', 'nwound', 'INT_LOG',
       'INT_IDEO', 'INT_MISC', 'INT_ANY', 'related'],
      dtype='object')

In [6]:
terrorism.isna().sum()

eventid                0
iyear                  0
imonth                 0
iday                   0
extended               0
country                0
country_txt            0
region                 0
region_txt             0
provstate            116
city                  93
latitude            2849
longitude           2849
specificity            3
vicinity               0
summary            36962
crit1                  0
crit2                  0
crit3                  0
doubtterr              0
alternative        70785
alternative_txt    70785
multiple               1
success                0
suicide                0
attacktype1            0
attacktype1_txt        0
gname                  0
motive             65345
individual             0
claimed            36953
nkill               6333
nwound             10783
INT_LOG                0
INT_IDEO               0
INT_MISC               0
INT_ANY                0
related            70960
dtype: int64

In [7]:
#Date Parsing code
def date_str(df):
    return (df.iyear.astype(str)+ "/" + df.imonth.astype(str) + "/" + df.iday.astype(str))

terrorism["date"] = (
    terrorism
    .pipe(date_str)
    .pipe(pd.to_datetime, format=f"%Y/%#m/%#d", errors="coerce")
)

terrorism =(
    terrorism
    .dropna(subset="date")
    .drop(columns=["imonth", "iday"])
) 



In [8]:
cities = pd.read_csv(f"Global Terrorism Dataset/worldcitiespop.csv")

cities["City"] =(
    cities["City"]
    .str.lower()
    .str.strip()
) 
cities = cities.drop_duplicates(subset=["City"])

terrorism["city"] =(
    terrorism["city"]
    .str.lower()
    .str.strip()
) 

terrorism = pd.merge(terrorism, cities, how="left", left_on="city", right_on="City")

terrorism["latitude"] = terrorism.latitude.fillna(terrorism.Latitude)
terrorism["longitude"] = terrorism.longitude.fillna(terrorism.Longitude)


C:\Users\HP\AppData\Local\Temp\ipykernel_16664\279449341.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  cities = pd.read_csv(f"Global Terrorism Dataset/worldcitiespop.csv")


In [9]:
drops = ["Country","City","AccentCity","Region","Population","Latitude","Longitude"]
terrorism = terrorism.drop(columns= drops)

In [10]:
analytics =(
    terrorism
    .groupby(["gname", "country_txt"], as_index=False)
    .filter(lambda x:x["nkill"].sum() > 20 )
    .groupby(["gname", "country_txt"], as_index=False)
    .count()
    .sort_values("nkill", ascending=False)
    .sort_values(by="country_txt", kind="stable")
    .set_index(["country_txt", "gname"])
    
)
analytics_txt = analytics[["nkill","nwound"]].to_string()
with open(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\analytics.txt", "w") as f:
    f.write(analytics_txt)
    f.close()


In [11]:
#Cleaning of Uppsala Uni Dataset


georef = pd.read_csv("Global Terrorism Dataset\Georef.csv\GEDEvent_v26_1.csv")

filtered = georef.loc[:,["year", "conflict_name", "country", "country_id", "region", "deaths_civilians", "best"]]

filtered.country = (
    filtered.country
    .str.lower()
    .str.split("(")
    .str[0]
    .str.strip()
    
)

civilians_deaths = (
    filtered
    .groupby("country")
    .sum()
)
civilians_deaths.sort_values("deaths_civilians",ascending=False)

C:\Users\HP\AppData\Local\Temp\ipykernel_16664\3055101714.py:4: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  georef = pd.read_csv("Global Terrorism Dataset\Georef.csv\GEDEvent_v26_1.csv")


,year,conflict_name,country_id,region,deaths_civilians,best
country,,,,,,
rwanda,5200247,Government of Rwanda - CiviliansRwanda: Govern...,1347819,AfricaAfricaAfricaAfricaAfricaAfricaAfricaAfri...,786327,794133
syria,177892951,Israel: PalestineGovernment of Syria - Civilia...,57564428,Middle EastMiddle EastMiddle EastMiddle EastMi...,125399,416870
sudan,10638945,Sudan: GovernmentArab - ZaghawaSudan: Governme...,3301875,AfricaAfricaAfricaAfricaAfricaAfricaAfricaAfri...,100590,195546
dr congo,20325434,Government of DR Congo (Zaire) - CiviliansGove...,4938220,AfricaAfricaAfricaAfricaAfricaAfricaAfricaAfri...,95426,150071
burundi,6384770,Burundi: GovernmentBurundi: GovernmentGovernme...,1645008,AfricaAfricaAfricaAfricaAfricaAfricaAfricaAfri...,35941,50952
...,...,...,...,...,...,...
trinidad and tobago,5970,Trinidad and Tobago: GovernmentTrinidad and To...,156,AmericasAmericasAmericas,0,39
qatar,4029,Russia (Soviet Union): ChechnyaIsrael: Palestine,1388,Middle EastMiddle East,0,6
malta,4020,Israel: PalestineGovernment of Libya - Civilians,676,EuropeEurope,0,0


In [12]:
civilians_deaths = (
    filtered
    .groupby(["country", "year"], as_index=False)
    .sum()
)
def reindex_list(df):
    reindexed = (
        df
        .groupby("country", as_index=False)
        .sum()
        .sort_values("deaths_civilians", ascending = False)
        .country
    )
    return reindexed

civilians_deaths = (
    civilians_deaths
    .sort_values("deaths_civilians", ascending=False)
    .sort_values("country", kind="stable")
    .set_index(["country", "year"])
    .pipe(lambda d:d.reindex(reindex_list(civilians_deaths), level="country"))
    .reset_index()
    .loc[:,["country", "year", "deaths_civilians"]]
)
terrorism

terrorism = (
    terrorism
    .drop(columns = "country")
    .rename(columns={"iyear":"year", "country_txt":"country"})
)

terrorism["country"] = (
    terrorism["country"]
    .str.lower()
    .str.strip()
)




merge_tryout = pd.merge(left=terrorism, right=civilians_deaths, on=["country", "year"], how="left")


In [ ]:
merge_tryout

0        united states
1              uruguay
2        united states
3        united states
4        united states
             ...      
87046      philippines
87047      afghanistan
87048          somalia
87049            syria
87050      philippines
Name: country, Length: 87051, dtype: object